# Daily Challenge: Classifying Handwritten Digits with CNNs

In this notebook we build and compare two architectures on the classic MNIST dataset:
1. A **Fully Connected Neural Network** (dense layers only)
2. A **Convolutional Neural Network (CNN)**

We'll see why spatial structure matters for image tasks.

## Step 1 — Load the MNIST dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

# Load raw data
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

print('Training images shape :', X_train_raw.shape)   # (60000, 28, 28)
print('Training labels shape :', y_train_raw.shape)   # (60000,)
print('Test images shape     :', X_test_raw.shape)    # (10000, 28, 28)
print('Test labels shape     :', y_test_raw.shape)    # (10000,)
print('Pixel value range     :', X_train_raw.min(), '–', X_train_raw.max())
print('Number of classes     :', len(np.unique(y_train_raw)))

In [ ]:
# Quick look at 10 sample digits
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train_raw[i], cmap='gray')
    ax.set_title(str(y_train_raw[i]))
    ax.axis('off')
plt.suptitle('Sample MNIST images', y=1.02)
plt.tight_layout()
plt.show()

## Step 2 — Preprocess for the Fully Connected Network

Dense layers expect a 1-D feature vector, so we **flatten** each 28×28 image to 784 values, then **normalise** pixel intensities to [0, 1].

In [ ]:
NUM_CLASSES = 10

# Flatten: (60000, 28, 28) → (60000, 784)
X_train_fc = X_train_raw.reshape(-1, 28 * 28).astype('float32') / 255.0
X_test_fc  = X_test_raw.reshape(-1, 28 * 28).astype('float32') / 255.0

# One-hot encode labels: 7 → [0,0,0,0,0,0,0,1,0,0]
y_train_cat = to_categorical(y_train_raw, NUM_CLASSES)
y_test_cat  = to_categorical(y_test_raw,  NUM_CLASSES)

print('FC input shape :', X_train_fc.shape)     # (60000, 784)
print('Label shape    :', y_train_cat.shape)    # (60000, 10)
print('Example label  :', y_train_cat[0])       # one-hot vector

## Step 3 — Build and train the Fully Connected Network

In [ ]:
# --- Architecture ---
# Input: 784 pixel values
# Hidden 1: 512 neurons, ReLU  → learns general patterns
# Dropout 0.3                  → regularisation, reduces overfitting
# Hidden 2: 256 neurons, ReLU  → deeper abstraction
# Dropout 0.3
# Output:   10 neurons, Softmax → probability distribution over digits

fc_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name='FullyConnected')

fc_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fc_model.summary()

In [ ]:
# Train
fc_history = fc_model.fit(
    X_train_fc, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,   # 10% of training data used for validation
    verbose=1
)

fc_test_loss, fc_test_acc = fc_model.evaluate(X_test_fc, y_test_cat, verbose=0)
print(f'\nFC Network — Test accuracy: {fc_test_acc:.4f}  |  Test loss: {fc_test_loss:.4f}')

In [ ]:
# Plot FC training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(fc_history.history['accuracy'],     label='Train')
axes[0].plot(fc_history.history['val_accuracy'], label='Validation')
axes[0].set_title('FC Network — Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(fc_history.history['loss'],     label='Train')
axes[1].plot(fc_history.history['val_loss'], label='Validation')
axes[1].set_title('FC Network — Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4 — Preprocess for the CNN

Conv2D layers expect a 4-D tensor: `(samples, height, width, channels)`.  
MNIST images are greyscale → 1 channel.

In [ ]:
# Reshape: (60000, 28, 28) → (60000, 28, 28, 1)
X_train_cnn = X_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test_cnn  = X_test_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Labels are the same one-hot vectors as before
print('CNN input shape:', X_train_cnn.shape)    # (60000, 28, 28, 1)
print('Label shape    :', y_train_cat.shape)    # (60000, 10)

## Step 5 — Build and train the CNN

In [ ]:
# --- Architecture ---
# Conv2D(32, 3×3) → learns local edge/curve detectors
# MaxPool2D       → halves spatial size, keeps dominant features
# Conv2D(64, 3×3) → combines edge detectors into digit-part detectors
# MaxPool2D
# Conv2D(64, 3×3) → high-level digit-shape detectors
# Flatten         → convert 3-D feature maps to 1-D vector
# Dense(64, ReLU) → combine spatial features globally
# Dropout(0.5)    → regularisation
# Dense(10, Softmax) → class probabilities

cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # Block 1
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
    layers.MaxPool2D(pool_size=(2, 2)),

    # Block 2
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    layers.MaxPool2D(pool_size=(2, 2)),

    # Block 3 — no pooling to preserve remaining spatial info
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),

    # Classification head
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name='CNN')

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# Train
cnn_history = cnn_model.fit(
    X_train_cnn, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

cnn_test_loss, cnn_test_acc = cnn_model.evaluate(X_test_cnn, y_test_cat, verbose=0)
print(f'\nCNN — Test accuracy: {cnn_test_acc:.4f}  |  Test loss: {cnn_test_loss:.4f}')

In [ ]:
# Plot CNN training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cnn_history.history['accuracy'],     label='Train')
axes[0].plot(cnn_history.history['val_accuracy'], label='Validation')
axes[0].set_title('CNN — Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(cnn_history.history['loss'],     label='Train')
axes[1].plot(cnn_history.history['val_loss'], label='Validation')
axes[1].set_title('CNN — Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6 — Compare the two models

In [ ]:
# Side-by-side accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Training accuracy ---
axes[0].plot(fc_history.history['accuracy'],      label='FC train',       color='steelblue',  linestyle='--')
axes[0].plot(fc_history.history['val_accuracy'],  label='FC val',         color='steelblue')
axes[0].plot(cnn_history.history['accuracy'],     label='CNN train',      color='darkorange', linestyle='--')
axes[0].plot(cnn_history.history['val_accuracy'], label='CNN val',        color='darkorange')
axes[0].set_title('Accuracy: FC vs CNN')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# --- Test accuracy bar chart ---
names  = ['Fully Connected', 'CNN']
scores = [fc_test_acc, cnn_test_acc]
colors = ['steelblue', 'darkorange']
bars = axes[1].bar(names, scores, color=colors, width=0.4)
for bar, score in zip(bars, scores):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.001,
                 f'{score:.4f}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylim(0.97, 1.0)
axes[1].set_title('Test Accuracy Comparison')
axes[1].set_ylabel('Test Accuracy')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Fully Connected test accuracy : {fc_test_acc:.4f} ({fc_test_acc*100:.2f}%)')
print(f'CNN test accuracy             : {cnn_test_acc:.4f} ({cnn_test_acc*100:.2f}%)')
print(f'Improvement (CNN – FC)        : {(cnn_test_acc - fc_test_acc)*100:.2f} percentage points')

In [ ]:
# Visualise CNN predictions on 16 test images
predictions = cnn_model.predict(X_test_cnn[:16], verbose=0).argmax(axis=1)
true_labels = y_test_raw[:16]

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
axes = axes.flatten()
for i in range(16):
    axes[i].imshow(X_test_raw[i], cmap='gray')
    color = 'green' if predictions[i] == true_labels[i] else 'red'
    axes[i].set_title(f'P:{predictions[i]} T:{true_labels[i]}', color=color, fontsize=9)
    axes[i].axis('off')

plt.suptitle('CNN Predictions (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.show()

## Summary & Key Takeaways

| | Fully Connected | CNN |
|---|---|---|
| **Input format** | Flat 784-dim vector | 28×28×1 spatial grid |
| **Spatial awareness** | ❌ — pixels treated independently | ✅ — local filters share weights |
| **Parameter efficiency** | Higher (all pixels connected) | Lower (small filters reused everywhere) |
| **Typical test accuracy** | ~97–98% | ~99%+ |

**Why CNNs win on images:**
- **Local connectivity** — a 3×3 filter looks at a neighbourhood, not the whole image at once, so it detects edges and curves regardless of where they appear.
- **Weight sharing** — the same filter is slid over every position, drastically reducing parameters and enforcing translation invariance.
- **Hierarchical features** — stacked Conv layers compose edges → parts → full digit shapes, mimicking the primate visual cortex.

**Fully connected networks** can still reach high accuracy on MNIST because the digit shapes are centred and small, but they fail to generalise when objects appear at different positions or scales — exactly the problem CNNs were designed to solve.